In [0]:
from pyspark.sql import functions as F

# =========================
# CONFIG
# =========================
STORAGE_ACCOUNT = "ragadziyada"
STORAGE_KEY = "qIXjwo7EGK8an84BPCk446tqY9L7K7r3a9W2WB+voe2vUCvW1SK3qc/7UGOicKyBAtrptYcVfTB7+AStvWZi0A=="

CURATED_CONTAINER = "curated-p1"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

CUSTOMER_FEATURES_IN = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customer_features/"
ARTICLE_FEATURES_IN = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/article_features/"
INTERACTIONS_IN = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customer_article_interactions/"

# =========================
# LOAD AVAILABLE CURATED DATA
# =========================
customer_features = spark.read.parquet(CUSTOMER_FEATURES_IN)
article_features = spark.read.parquet(ARTICLE_FEATURES_IN)
interactions = spark.read.parquet(INTERACTIONS_IN)

print("customer_features rows:", customer_features.count())
print("article_features rows:", article_features.count())
print("interactions rows:", interactions.count())

print("\n=== customer_features schema ===")
customer_features.printSchema()

print("\n=== article_features schema ===")
article_features.printSchema()

print("\n=== interactions schema ===")
interactions.printSchema()

# =========================
# EDA 1: TOP PRODUCT TYPES
# =========================
print("\n=== Top product types by count ===")
display(
    interactions.groupBy("product_type_name")
      .count()
      .orderBy(F.desc("count"))
      .limit(20)
)

# =========================
# EDA 2: TOP DEPARTMENTS
# =========================
print("\n=== Top departments ===")
display(
    interactions.groupBy("department_name")
      .count()
      .orderBy(F.desc("count"))
      .limit(20)
)

# =========================
# EDA 3: SALES CHANNEL SPLIT
# =========================
print("\n=== Sales channel split ===")
display(
    interactions.groupBy("sales_channel_id")
      .count()
      .orderBy("sales_channel_id")
)

# =========================
# EDA 4: TRANSACTIONS OVER TIME
# =========================
print("\n=== Transactions over time ===")
display(
    interactions.groupBy("year", "month")
      .count()
      .orderBy("year", "month")
)

# =========================
# EDA 5: CUSTOMER FEATURE SUMMARY
# =========================
print("\n=== Customer feature summary ===")
display(
    customer_features.select(
        F.avg("customer_purchase_count").alias("avg_purchase_count"),
        F.avg("customer_total_spend").alias("avg_total_spend"),
        F.avg("customer_avg_price").alias("avg_price"),
        F.avg("customer_recency_days").alias("avg_recency_days"),
        F.avg("customer_avg_basket_size").alias("avg_basket_size")
    )
)

# =========================
# EDA 6: ARTICLE FEATURE SUMMARY
# =========================
print("\n=== Article feature summary ===")
display(
    article_features.select(
        F.avg("article_total_sales").alias("avg_total_sales"),
        F.avg("article_unique_customers").alias("avg_unique_customers"),
        F.avg("article_avg_price").alias("avg_article_price"),
        F.avg("article_sales_last_7d").alias("avg_sales_last_7d"),
        F.avg("article_sales_last_30d").alias("avg_sales_last_30d")
    )
)

# =========================
# EDA 7: TOP ARTICLES BY SALES
# =========================
print("\n=== Top articles by total sales ===")
display(
    article_features.orderBy(F.desc("article_total_sales"))
      .limit(20)
)

print("\nEDA finished successfully.")

customer_features rows: 1362281
article_features rows: 104547
interactions rows: 28813419

=== customer_features schema ===
root
 |-- customer_id: string (nullable = true)
 |-- customer_purchase_count: long (nullable = true)
 |-- customer_total_spend: double (nullable = true)
 |-- customer_avg_price: double (nullable = true)
 |-- customer_unique_articles: long (nullable = true)
 |-- customer_unique_product_types: long (nullable = true)
 |-- customer_unique_departments: long (nullable = true)
 |-- customer_last_purchase_date: date (nullable = true)
 |-- customer_first_purchase_date: date (nullable = true)
 |-- customer_recency_days: integer (nullable = true)
 |-- customer_purchase_frequency: double (nullable = true)
 |-- customer_preferred_department: string (nullable = true)
 |-- customer_preferred_index_group: string (nullable = true)
 |-- customer_avg_basket_size: double (nullable = true)


=== article_features schema ===
root
 |-- article_id: string (nullable = true)
 |-- article_to

product_type_name,count
Trousers,3725520
Dress,2918786
Sweater,2614740
T-shirt,2003299
Top,1465853
Blouse,1389973
Vest top,1258605
Bra,1237163
Shorts,1041512
Bikini top,961314



=== Top departments ===


department_name,count
Swimwear,2128005
Trouser,1549348
Blouse,1519154
Knitwear,1505655
Jersey,1390422
Jersey Basic,1301964
Expressive Lingerie,1084735
Jersey fancy,1078516
Basic 1,1027967
Dress,1008274



=== Sales channel split ===


...


=== Article feature summary ===


avg_total_sales,avg_unique_customers,avg_article_price,avg_sales_last_7d,avg_sales_last_30d
275.60254239719933,261.18816417496436,0.02869056034789185,2.3101093288186174,10.082508345528804



=== Top articles by total sales ===


article_id,article_total_sales,article_unique_customers,article_avg_price,article_last_sold_date,article_sales_last_7d,article_sales_last_30d,product_type_name,department_name,index_group_name,colour_group_name,garment_group_name,article_popularity_rank
0706016001,42672,32251,0.03238685392080181,2020-09-22,335,2095,Trousers,Trousers,Divided,Black,Trousers,1
0706016002,30862,25485,0.03239025887795717,2020-09-22,238,910,Trousers,Trousers,Divided,Light Blue,Trousers,2
0372860001,29337,25559,0.012960742580520955,2020-09-22,236,1085,Socks,Shopbasket Socks,Ladieswear,Black,Socks and Tights,3
0610776002,25234,22571,0.00807677494582906,2020-09-22,175,955,T-shirt,Jersey Basic,Ladieswear,Black,Jersey Basic,4
0759871002,23799,21613,0.0056075493842854375,2020-09-21,20,186,Vest top,EQ Divided Basics,Divided,Black,Jersey Basic,5
0372860002,22472,20038,0.012182521676692986,2020-09-22,238,1100,Socks,Shopbasket Socks,Ladieswear,White,Socks and Tights,6
0464297007,21782,18554,0.016172152718229478,2020-09-22,32,375,Underwear bottom,Casual Lingerie,Ladieswear,Black,"Under-, Nightwear",7
0399223001,19604,15929,0.030976297642473522,2020-09-22,34,191,Trousers,Denim Trousers,Divided,Black,Trousers Denim,8
0720125001,18975,17611,0.03241261159866908,2020-09-22,67,354,Leggings/Tights,Ladies Sport Bottoms,Sport,Black,Jersey Fancy,9
0610776001,18777,16854,0.008068033105774001,2020-09-22,115,564,T-shirt,Jersey Basic,Ladieswear,White,Jersey Basic,10



EDA finished successfully.
